In [1]:
!pip install selenium --upgrade
!pip install requests
from selenium import webdriver
from selenium.webdriver.common.by import By

import sys
!sudo add-apt-repository ppa:saiarcot895/chromium-beta
!sudo apt remove chromium-browser
!sudo snap remove chromium
!sudo apt install chromium-browser

!pip install tqdm
import pandas as pd
from tqdm import tqdm
!pip install selenium
!apt-get update
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service

sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')

options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_experimental_option("detach", True)
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

webdriver_service = Service('/usr/lib/chromium-browser/chromedriver')
driver = webdriver.Chrome(service=webdriver_service,options = options)

from time import sleep

from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup
import requests
import re
import random
import requests
import json
from datetime import datetime, timedelta
import os
import csv

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 141.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 17.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
PPA publishes dbgsym, you may need to include 'main/debug' component
Repository: 'deb https://ppa.launchpadcontent.net/saiarcot895/chromium-beta/ubuntu/ jammy main'
Description:
This PPA contains the latest Chromium Beta builds, with hardware video decoding enabled (hidden behind a flag), and support for Widevine (needed for viewing many DRM-protected videos) enabled.

== Hardware Video Decoding ==

To enable hardware video decoding, start Chromium with the --enable-features=VaapiVideoDecoder argument. To make this persistent, create a file at /etc/chromium-browser/customizations/92-vaapi-hardware

In [2]:
import re
import csv
from urllib.parse import urljoin

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    TimeoutException,
)

In [3]:
LIST_URL = "https://www.chosun.com/opinion/editorial/?page={page}"
DOMAIN = "https://www.chosun.com"

TARGET_YEAR = 2025
MAX_PAGES = 1000

page = 1
collected_articles = []
seen_urls = set()



In [4]:
def find_article_card(driver, date_element):
    """
    날짜 요소와 제목 링크가 각각 하나씩 포함된
    가장 가까운 상위 기사 카드 요소를 찾습니다.
    """
    return driver.execute_script(
        """
        let element = arguments[0];

        while (element && element !== document.body) {
            const dates = element.querySelectorAll(
                "div.text.text--wrap-pre"
            );

            const headlines = element.querySelectorAll(
                "a.story-card__headline"
            );

            if (dates.length === 1 && headlines.length === 1) {
                return element;
            }

            element = element.parentElement;
        }

        return null;
        """,
        date_element,
    )


while page <= MAX_PAGES:
    current_url = LIST_URL.format(page=page)
    driver.get(current_url)

    print(f"\n[페이지 {page}] {current_url}")

    try:
        # 페이지 내 날짜 요소가 나타날 때까지 대기
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div.text.text--wrap-pre")
            )
        )

    except TimeoutException:
        print("날짜 요소를 찾지 못해 탐색을 종료합니다.")
        break

    date_elements = driver.find_elements(
        By.CSS_SELECTOR,
        "div.text.text--wrap-pre",
    )

    page_years = []
    page_collected_count = 0

    for date_element in date_elements:
        try:
            date_text = date_element.text.strip()

            # 예: 2026.07.09(목)
            year_match = re.match(r"^(\d{4})\.", date_text)

            if not year_match:
                continue

            year = int(year_match.group(1))
            page_years.append(year)

            # 2025년이 아니면 수집하지 않음
            if year != TARGET_YEAR:
                continue

            # 현재 날짜 요소가 포함된 기사 카드 찾기
            article_card = find_article_card(driver, date_element)

            if article_card is None:
                print(f"[카드 탐색 실패] {date_text}")
                continue

            # 같은 카드 내부에서 제목 링크 찾기
            headline_element = article_card.find_element(
                By.CSS_SELECTOR,
                "a.story-card__headline",
            )

            href = headline_element.get_attribute("href")
            title = headline_element.text.strip()

            if not href:
                continue

            # 상대 경로와 절대 경로 모두 처리
            article_url = urljoin(DOMAIN, href)

            # 중복 URL 제거
            if article_url in seen_urls:
                continue

            seen_urls.add(article_url)

            collected_articles.append(
                {
                    "date": date_text,
                    "title": title,
                    "url": article_url,
                }
            )

            page_collected_count += 1

            print(f"[수집] {date_text}")
            print(f"       {title}")
            print(f"       {article_url}")

        except (
            NoSuchElementException,
            StaleElementReferenceException,
        ) as error:
            print(f"[기사 처리 실패] {error}")
            continue

    print(
        f"페이지 내 날짜 수: {len(date_elements)}, "
        f"2025년 수집 수: {page_collected_count}"
    )

    # 날짜를 전혀 찾지 못한 경우 종료
    if not page_years:
        print("유효한 날짜가 없어 탐색을 종료합니다.")
        break

    # 페이지의 가장 최신 연도도 2024 이하라면
    # 이후 페이지에는 2025년 기사가 없다고 판단
    if max(page_years) < TARGET_YEAR:
        print(
            f"현재 페이지의 가장 최신 연도가 {max(page_years)}년이므로 "
            "탐색을 종료합니다."
        )
        break

    page += 1


print("\n수집 완료")
print(f"총 수집 기사 수: {len(collected_articles)}")


[페이지 1] https://www.chosun.com/opinion/editorial/?page=1
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 2] https://www.chosun.com/opinion/editorial/?page=2
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 3] https://www.chosun.com/opinion/editorial/?page=3
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 4] https://www.chosun.com/opinion/editorial/?page=4
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 5] https://www.chosun.com/opinion/editorial/?page=5
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 6] https://www.chosun.com/opinion/editorial/?page=6
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 7] https://www.chosun.com/opinion/editorial/?page=7
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 8] https://www.chosun.com/opinion/editorial/?page=8
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 9] https://www.chosun.com/opinion/editorial/?page=9
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 10] https://www.chosun.com/opinion/editorial/?page=10
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 11] https://www.chosun.com/opinion/editorial/?page=11
페이지 내 날짜 수: 20, 2025년 수집 수: 0

[페이지 12] https://www.chosun

In [5]:
output_path = "조선일보_사설_2025_urls.csv"

with open(
    output_path,
    "w",
    newline="",
    encoding="utf-8-sig",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["date", "title", "url"],
    )

    writer.writeheader()
    writer.writerows(collected_articles)

print(f"저장 완료: {output_path}")

저장 완료: 조선일보_사설_2025_urls.csv


In [6]:
# 1. 파일 및 크롤링 설정
# =========================================================

INPUT_PATH = "조선일보_사설_2025_urls.csv"
OUTPUT_PATH = "조선일보_사설_2025_articles.csv"

# 이전 실행 결과를 중간 저장할 파일
CHECKPOINT_PATH = "chosun_editorial_2025_checkpoint.csv"

WAIT_TIME = 15
REQUEST_INTERVAL = 1.5

# 전체 클래스명을 모두 사용하지 않고,
# 본문을 구분하는 핵심 클래스만 사용
ARTICLE_BODY_SELECTOR = (
    "p.article-body__content.article-body__content-text"
)



In [ ]:
# 3. 텍스트 정제 함수

def clean_text(text):
    """
    줄바꿈, 탭, 연속 공백을 하나의 공백으로 변경합니다.
    """
    if not isinstance(text, str):
        return ""

    return re.sub(r"\s+", " ", text).strip()

# 4. 기사 본문 수집 함수
def collect_article_content(driver, url):
    """
    기사 URL에 접속하여 본문 p 태그의 모든 텍스트를 수집합니다.

    예:
    <p>A</p>
    <p>B</p>

    결과:
    A B
    """
    driver.get(url)

    # 본문 요소가 하나 이상 나타날 때까지 대기
    WebDriverWait(driver, WAIT_TIME).until(
        EC.presence_of_all_elements_located(
            (
                By.CSS_SELECTOR,
                ARTICLE_BODY_SELECTOR,
            )
        )
    )

    paragraph_elements = driver.find_elements(
        By.CSS_SELECTOR,
        ARTICLE_BODY_SELECTOR,
    )

    paragraph_texts = []

    for paragraph_element in paragraph_elements:
        try:
            text = clean_text(paragraph_element.text)

            if text:
                paragraph_texts.append(text)

        except StaleElementReferenceException:
            continue

    # 각 문단을 하나의 공백으로 연결
    article_content = " ".join(paragraph_texts)

    return clean_text(article_content), len(paragraph_texts)


# 5. 기존 URL 목록 불러오기
url_df = pd.read_csv(
    INPUT_PATH,
    encoding="utf-8-sig",
)

required_columns = {"date", "title", "url"}
missing_columns = required_columns - set(url_df.columns)

if missing_columns:
    raise ValueError(
        f"입력 CSV에 필요한 열이 없습니다: {missing_columns}"
    )

# URL 중복 제거
url_df = (
    url_df
    .drop_duplicates(subset=["url"])
    .reset_index(drop=True)
)

print(f"전체 수집 대상 기사 수: {len(url_df)}")


# =========================================================
# 6. 체크포인트 불러오기
# =========================================================

if os.path.exists(CHECKPOINT_PATH):
    result_df = pd.read_csv(
        CHECKPOINT_PATH,
        encoding="utf-8-sig",
    )

    completed_urls = set(
        result_df.loc[
            result_df["status"] == "success",
            "url",
        ].dropna()
    )

    print(
        f"체크포인트를 불러왔습니다. "
        f"수집 완료 기사 수: {len(completed_urls)}"
    )

else:
    result_df = pd.DataFrame(
        columns=[
            "date",
            "title",
            "url",
            "content",
            "paragraph_count",
            "status",
            "error",
        ]
    )

    completed_urls = set()


# =========================================================
# 7. 각 기사 URL에 접속해 본문 수집
# =========================================================

try:
    for index, row in url_df.iterrows():
        date = row["date"]
        title = row["title"]
        url = row["url"]

        # 이미 정상적으로 수집한 기사는 건너뜀
        if url in completed_urls:
            print(
                f"[건너뜀] {index + 1}/{len(url_df)} "
                f"{title}"
            )
            continue

        print("\n" + "=" * 80)
        print(f"[기사 수집] {index + 1}/{len(url_df)}")
        print(f"날짜: {date}")
        print(f"제목: {title}")
        print(f"URL: {url}")

        content = ""
        paragraph_count = 0
        status = "failed"
        error_message = ""

        try:
            content, paragraph_count = collect_article_content(
                driver,
                url,
            )

            if content:
                status = "success"

                print(
                    f"[수집 완료] "
                    f"문단 수: {paragraph_count}, "
                    f"글자 수: {len(content)}"
                )

                print(f"본문 일부: {content[:150]}...")

            else:
                status = "empty"
                error_message = "본문 요소는 존재하지만 텍스트가 없습니다."

                print("[본문 없음] 추출된 텍스트가 없습니다.")

        except TimeoutException:
            status = "timeout"
            error_message = "본문 요소 로딩 시간 초과"

            print("[시간 초과] 본문 요소를 찾지 못했습니다.")

        except WebDriverException as error:
            status = "webdriver_error"
            error_message = str(error)

            print(f"[WebDriver 오류] {error}")

        except Exception as error:
            status = "failed"
            error_message = str(error)

            print(f"[수집 실패] {error}")

        new_row = pd.DataFrame(
            [
                {
                    "date": date,
                    "title": title,
                    "url": url,
                    "content": content,
                    "paragraph_count": paragraph_count,
                    "status": status,
                    "error": error_message,
                }
            ]
        )

        # 이전 실패 기록이 있다면 동일 URL의 과거 기록 제거
        result_df = result_df[
            result_df["url"] != url
        ]

        result_df = pd.concat(
            [result_df, new_row],
            ignore_index=True,
        )

        # 매 기사 처리 후 체크포인트 저장
        result_df.to_csv(
            CHECKPOINT_PATH,
            index=False,
            encoding="utf-8-sig",
        )

        time.sleep(REQUEST_INTERVAL)

finally:
    driver.quit()


# =========================================================
# 8. 최종 데이터프레임 정리 및 저장
# =========================================================

result_df = result_df[
    [
        "date",
        "title",
        "url",
        "content",
        "paragraph_count",
        "status",
        "error",
    ]
].copy()

result_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 80)
print("본문 수집이 완료되었습니다.")
print(f"전체 처리 기사 수: {len(result_df)}")
print(
    "정상 수집 기사 수:",
    (result_df["status"] == "success").sum(),
)
print(
    "수집 실패 기사 수:",
    (result_df["status"] != "success").sum(),
)
print(f"최종 저장 파일: {OUTPUT_PATH}")

# 수집 결과 확인
display(
    result_df[
        ["date", "title", "url", "content"]
    ].head()
)

전체 수집 대상 기사 수: 911

[기사 수집] 1/911
날짜: 2025.12.31(수)
제목: [사설] 민주당 '공천 헌금' 의혹, 강선우 뿐인가
URL: https://www.chosun.com/opinion/editorial/2025/12/31/6BZKC52WQFFC7JFOKLZTB6TDOY/
[수집 완료] 문단 수: 4, 글자 수: 1197
본문 일부: 민주당 강선우 의원이 2022년 지방선거를 앞두고 ‘보좌관이 지역구 시의원 후보에게서 1억원을 받아 보관 중’이라는 취지로 말하는 음성 녹음이 공개됐다. 강 의원은 당시 서울시당 공천관리위원회 간사를 맡은 김병기 의원을 만나 “제가 어떻게 하면 되느냐”고 물었다. 돈을...

[기사 수집] 2/911
날짜: 2025.12.31(수)
제목: [사설] 검찰은 서해 피살 유족의 '항소' 호소 외면 말아야
URL: https://www.chosun.com/opinion/editorial/2025/12/31/3KJK5JGZNZESXNRPGNXHNHONLU/
[수집 완료] 문단 수: 5, 글자 수: 1210
본문 일부: 이재명 대통령이 30일 최근 1심 무죄 판결이 나온 ‘서해 공무원 피살 은폐’에 대해 “없는 사건을 만들고, 있는 증거를 숨기고, 사람을 감옥에 보내려고 시도하는 게 말이 되느냐”며 “책임을 묻든지 해야 할 것 같다”고 했다. 김민석 총리도 “사실상 조작 기소로 볼 수...

[기사 수집] 3/911
날짜: 2025.12.31(수)
제목: [사설] 이 대통령, 공공개혁 완수하면 큰 업적 될 것
URL: https://www.chosun.com/opinion/editorial/2025/12/31/JMR5NS5ZAZAKDFPNER6CT5CQXY/
[수집 완료] 문단 수: 4, 글자 수: 907
본문 일부: 이재명 대통령이 국무회의에서 “공공기관을 어떻게 개혁할지, 통폐합과 신설을 포함해 속도를 내달라”고 지시했다. 공공기관들이 “하는 일이 뭔지도, 뭘 해야 하는지도 모르고 시간이나 때우고 누릴 것만